# 3. Multi-output neural network, SHAP and in-range screening

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ddmo.data import FEATURES, TARGETS, design_matrix, load_films, variance_inflation
summary = json.loads((ROOT / "results" / "summary.json").read_text())
df = load_films()
X, Y = design_matrix(df)
pd.set_option("display.precision", 3)
pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.width", 120)

In [2]:
s = summary["nested_cv"]["mlp"]
print(s["label"], "| nested CV R2 =", s["r2_mean"], "+/-", s["r2_std"])
pd.Series(s["r2_per_target"], name="R2 per target")

Multi-output neural network | nested CV R2 = 0.9881 +/- 0.0136


conductivity_S_per_m    0.998
tensile_strength_MPa    0.995
thickness_um            0.971
Name: R2 per target, dtype: float64

![Out-of-fold parity plots](../results/figures/parity_nested_cv.svg)

One network predicts all three properties from a shared hidden layer; hidden size and weight decay are chosen in the inner loop. It is most useful for thickness, where it raises out-of-sample R² from 0.89 (linear) to 0.97.

In [3]:
pd.DataFrame({m: pd.DataFrame(v).stack() for m, v in summary["shap_grouped_share"].items()}).round(3)

                                                              mlp  lasso
composition (CNC, CNF, rGO)           conductivity_S_per_m  0.852  0.883
                                      tensile_strength_MPa  0.827  0.827
                                      thickness_um          0.847  0.914
reduction chemistry (acid, pH, agent) conductivity_S_per_m  0.148  0.117
                                      tensile_strength_MPa  0.173  0.173
                                      thickness_um          0.153  0.086

![Share of mean |SHAP| per input, multi-output network](../results/figures/shap_importance_mlp.svg)

**SHAP.** Per feature, the network credits CNC more and LASSO credits rGO more, because CNC and rGO are correlated at 0.92 and the split of credit between them is not identifiable from these data. SHAP values are additive, so summing within groups gives an identifiable answer, and there the two models agree: composition carries about 83–91% of the attribution for every target and reduction chemistry the rest.

In [4]:
pd.Series(summary["screening"], dtype=object)

best_observed_index                                                                             57
best_observed_F                                                                             15.004
candidate                                        {'CNC_wt_pct': 1.9832, 'CNF_wt_pct': 0.5897, '...
predicted_F                                      {'mlp': 14.964967470431422, 'lasso': 14.989308...
predicted_targets                                {'mlp': [1.7079, 45.8981, 146.3351], 'lasso': ...
n_candidates                                                                                 20000
n_predicted_above_best_observed_by_all_models                                                    0
dtype: object

**Screening inside the measured region.** Candidates are convex combinations of measured films with the same reducing agent, which keeps the coupled pH and acid concentration realistic. Under the paper's objective F = 0.7σ + 0.3Ts, no candidate is predicted to beat the best measured film (L-ascorbic acid #28, F = 15.0) by both models. The best film sits on the edge of the measured design, so further gains require experiments beyond it. That is the setting for sequential design with calibrated uncertainty, developed in the materials-mobo repository.